# 07 — Explore ingested telemetry

Visualizza i dati arrivati in `kql_telemetry.raw_telemetry` dal simulator via Eventstream.
Stesso pattern dual-mode degli altri notebook: funziona sia nel runtime Fabric sia in locale.

Sezioni:
1. Setup + auth + client KQL
2. Ingestion freshness (per macchina, ultimi 5 min)
3. Time series ultimo intervallo (8 sensori, una macchina)
4. Confronto fra macchine su un sensore
5. Distribuzioni dei sensori (box plot)
6. Correlazioni multivariate (scatter)
7. Tail dei valori sospetti (|z| > 3)

Tutti i grafici sono renderizzati lato client (matplotlib) per restare portatili.

## 0. Install runtime dependencies (Fabric-only)

In [ ]:
%pip install -q azure-kusto-data azure-identity python-dotenv

## 1. Config

In [ ]:
MACHINE_FOCUS = 'M-001'
WINDOW_RECENT = '15m'        # for freshness + tail
WINDOW_PLOT   = '1h'         # for time series + box plots
SENSOR_FOCUS  = 'temperature_motor'

SENSORS = [
    'temperature_motor', 'temperature_bearing',
    'vibration_axial',   'vibration_radial',
    'current',           'spindle_rpm',
    'pressure_hydraulic','power',
]

## 2. Auth + Kusto client (dual mode)

In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table

FABRIC_API   = 'https://api.fabric.microsoft.com/v1'
FABRIC_SCOPE = 'https://api.fabric.microsoft.com/.default'

try:
    import notebookutils
    IN_FABRIC = True
except ImportError:
    IN_FABRIC = False

if IN_FABRIC:
    # Fabric Spark kernel: use the user's notebook identity via notebookutils.
    ctx        = notebookutils.runtime.context
    WORKSPACE  = ctx.get('currentWorkspaceName')
    KQLDB_NAME = os.environ.get('FABRIC_KQLDB_NAME', 'kql_telemetry')
    fabric_token = notebookutils.credentials.getToken('pbi')      # Fabric REST API
    kusto_token  = notebookutils.credentials.getToken('kusto')    # Eventhouse / KQL DB
    print(f'[auth] Fabric runtime (workspace={WORKSPACE})')
else:
    from dotenv import load_dotenv
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / '.env').exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent
    load_dotenv(REPO_ROOT / '.env')
    sys.path.insert(0, str(REPO_ROOT / 'tools'))
    from _fabric_auth import get_credential
    WORKSPACE  = os.environ['FABRIC_WORKSPACE_NAME']
    KQLDB_NAME = os.environ['FABRIC_KQLDB_NAME']
    cred       = get_credential(os.environ['FABRIC_TENANT_ID'], FABRIC_SCOPE, REPO_ROOT)
    fabric_token = cred.get_token(FABRIC_SCOPE).token
    kusto_token  = None  # built on-the-fly via KustoConnectionStringBuilder
    print(f'[auth] local .venv (.env @ {REPO_ROOT})')

# Resolve workspace + KQL DB metadata via Fabric REST.
h  = {'Authorization': f'Bearer {fabric_token}'}
ws = next(w for w in requests.get(f'{FABRIC_API}/workspaces', headers=h).json()['value']
          if w['displayName'] == WORKSPACE)
db = next(d for d in requests.get(f'{FABRIC_API}/workspaces/{ws["id"]}/kqlDatabases',
                                   headers=h).json()['value']
          if d['displayName'] == KQLDB_NAME)
query_uri = db['properties']['queryServiceUri']
print('queryServiceUri:', query_uri)

# Build Kusto client. Two paths because Fabric kernel has no msal token cache.
if IN_FABRIC:
    kcsb = KustoConnectionStringBuilder.with_aad_user_token_authentication(query_uri, kusto_token)
else:
    kcsb = KustoConnectionStringBuilder.with_azure_token_credential(query_uri, cred)
kusto = KustoClient(kcsb)

def kql(q: str) -> pd.DataFrame:
    return dataframe_from_result_table(kusto.execute(KQLDB_NAME, q).primary_results[0])

plt.rcParams['figure.figsize'] = (13, 3.4)
plt.rcParams['axes.grid'] = True

## 3. Ingestion freshness

Conteggio righe e ultimo timestamp per macchina (ultimi 15 minuti).
Se una macchina manca o `lag_s` cresce → il flusso si è fermato.

In [ ]:
fresh = kql(f'''
raw_telemetry
| where ts > ago({WINDOW_RECENT})
| summarize n = count(),
            latest_ts = max(ts),
            sensors = dcount(sensor_id)
          by machine_id
| extend lag_s = datetime_diff("second", now(), latest_ts)
| order by machine_id asc
''')
fresh

## 4. Time series — tutti gli 8 sensori della macchina di focus

In [ ]:
ts_df = kql(f'''
raw_telemetry
| where machine_id == '{MACHINE_FOCUS}' and ts > ago({WINDOW_PLOT})
| summarize value = avg(value) by ts = bin(ts, 5s), sensor_id
| order by ts asc
''')
wide = (ts_df.pivot_table(index='ts', columns='sensor_id', values='value')
              .reindex(columns=SENSORS))

fig, axes = plt.subplots(len(SENSORS), 1, figsize=(13, 1.6*len(SENSORS)), sharex=True)
for ax, s in zip(axes, SENSORS):
    if s in wide:
        ax.plot(wide.index, wide[s], lw=0.8, color='navy')
    ax.set_ylabel(s, fontsize=8)
axes[-1].set_xlabel('ts')
fig.suptitle(f'{MACHINE_FOCUS} — last {WINDOW_PLOT}', y=0.995)
plt.tight_layout(); plt.show()

## 5. Confronto fra macchine — stesso sensore

In [ ]:
cmp = kql(f'''
raw_telemetry
| where sensor_id == '{SENSOR_FOCUS}' and ts > ago({WINDOW_PLOT})
| summarize value = avg(value) by ts = bin(ts, 5s), machine_id
| order by ts asc
''')
cmp_wide = cmp.pivot_table(index='ts', columns='machine_id', values='value')

fig, ax = plt.subplots(figsize=(13, 4))
for col in cmp_wide.columns:
    ax.plot(cmp_wide.index, cmp_wide[col], lw=0.9, label=col)
ax.set_title(f'{SENSOR_FOCUS} — last {WINDOW_PLOT}')
ax.set_xlabel('ts'); ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

## 6. Distribuzioni dei sensori (box plot, solo running)

Esclude i campioni con `value == 0` su `temperature_motor` (proxy di stato OFF).

In [ ]:
dist = kql(f'''
let off_ts = raw_telemetry
    | where ts > ago({WINDOW_PLOT}) and sensor_id == 'temperature_motor' and value == 0
    | project ts, machine_id;
raw_telemetry
| where ts > ago({WINDOW_PLOT})
| join kind=anti off_ts on machine_id, ts
| project sensor_id, value
''')

if dist.empty:
    print('No running samples in window (machines still in OFF).')
else:
    fig, ax = plt.subplots(figsize=(13, 4))
    data = [dist.loc[dist.sensor_id == s, 'value'].values for s in SENSORS]
    ax.boxplot(data, labels=SENSORS, showfliers=True)
    ax.set_yscale('symlog')
    ax.set_title(f'Sensor distributions (running only) — last {WINDOW_PLOT}')
    plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

## 7. Correlazione multivariata (current vs power, load proxy)

In [ ]:
piv = kql(f'''
raw_telemetry
| where ts > ago({WINDOW_PLOT})
| summarize value = avg(value) by ts = bin(ts, 5s), machine_id, sensor_id
| evaluate pivot(sensor_id, any(value))
| where temperature_motor > 0
''')

if piv.empty:
    print('No pivot data yet (machines still in OFF).')
else:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].scatter(piv['current'], piv['power'], s=5, alpha=0.4)
    axes[0].set_xlabel('current (A)'); axes[0].set_ylabel('power (kW)')
    axes[0].set_title('power vs current')

    axes[1].scatter(piv['current'], piv['vibration_radial'], s=5, alpha=0.4, c='tab:orange')
    axes[1].set_xlabel('current (A)'); axes[1].set_ylabel('vibration_radial (g)')
    axes[1].set_title('vibration vs load proxy')

    axes[2].scatter(piv['temperature_motor'], piv['temperature_bearing'], s=5, alpha=0.4, c='tab:red')
    axes[2].set_xlabel('T_motor (°C)'); axes[2].set_ylabel('T_bearing (°C)')
    axes[2].set_title('bearing slaved to motor')
    plt.tight_layout(); plt.show()

## 8. Tail dei valori sospetti (|z| > 3, ultimi 15 min)

Z-score per `(machine, sensor)` calcolato su una finestra di 1 ora come baseline.
Aiuta a vedere a colpo d'occhio se l'`anomaly-prob` overlay del simulator ha sparato spike.

In [ ]:
spikes = kql(f'''
let base = raw_telemetry
    | where ts > ago(1h) and ts <= ago({WINDOW_RECENT})
    | summarize mu = avg(value), sigma = stdev(value) by machine_id, sensor_id;
raw_telemetry
| where ts > ago({WINDOW_RECENT})
| lookup base on machine_id, sensor_id
| where isnotnull(sigma) and sigma > 0
| extend z = (value - mu) / sigma
| where abs(z) > 3
| project ts, machine_id, sensor_id, value, mu, sigma, z, quality
| order by ts desc
| take 50
''')
spikes